# Nutritional Correlation Analysis (Strict Schema, PySpark)

This notebook computes and visualizes correlations between nutrients in foods using PySpark, with strict schema mapping. Structure and approach follow the nutrient similarity search notebook.

In [ ]:
# Imports and Spark session
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# In cluster use the HDFS path prefix
# path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
path_prefix = "" 

spark = SparkSession.builder.appName("NutritionalCorrelationStrict").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# Configuration
input_path = f"{path_prefix}/output/nutritional_profiles"
fallback_file = f"{path_prefix}/output/nutritional_profiles/part-00000-d5d8abf9-0202-406e-af46-ff13446ce22f-c000.snappy.parquet"
output_dir = f"{path_prefix}/output/NutritionalCorrelation"

# Strict schema mapping (edit as needed for your dataset)
schema_columns = {
    'id': 'fdc_id',
    'name': 'food_description',
    'category': 'food_type',
    'energy': 'energy',
    'protein': 'protein',
    'carb': 'carbs',
    'fat': 'total_fat',
    'fiber': 'fiber',
    'sugar': 'sugars',
    'saturated_fat': 'saturated_fat',
    'monounsaturated_fat': 'monounsaturated_fat',
    'polyunsaturated_fat': 'polyunsaturated_fat',
    'trans_fat': 'trans_fat',
    'cholesterol': 'cholesterol',
    'sodium': 'sodium',
    'calcium': 'calcium',
    'iron': 'iron',
    'potassium': 'potassium',
    'vitamin_c': 'vitamin_c',
    'vitamin_a': 'vitamin_a',
    'vitamin_b12': 'vitamin_b12',
}

# Nutrients to include in correlation
nutrient_cols = [
    'energy', 'protein', 'carb', 'fat', 'fiber', 'sugar', 'saturated_fat',
    'monounsaturated_fat', 'polyunsaturated_fat', 'trans_fat', 'cholesterol',
    'sodium', 'calcium', 'iron', 'potassium', 'vitamin_c', 'vitamin_a', 'vitamin_b12'
]

In [ ]:
# Data loading and schema mapping
def read_profiles(input_path, fallback_file):
    import os
    if os.path.isdir(input_path):
        try:
            df = spark.read.parquet(input_path)
        except Exception:
            df = None
    else:
        df = None
    if df is None and os.path.exists(fallback_file):
        df = spark.read.parquet(fallback_file)
    if df is None:
        raise FileNotFoundError(f"No parquet at {input_path} or {fallback_file}")
    return df

def apply_strict_schema(df, schema_map):
    # Select and rename columns according to schema
    exprs = []
    for k, src in schema_map.items():
        if src in df.columns:
            exprs.append(F.col(src).alias(k))
        else:
            exprs.append(F.lit(None).alias(k))
    return df.select(*exprs)

df_raw = read_profiles(input_path, fallback_file)
df = apply_strict_schema(df_raw, schema_columns)
df.cache(); print(f"Loaded {df.count()} rows.")
df.show(5)

In [ ]:
# Filter: Optionally restrict to a food category or apply text filters
# Example: df = df.filter(df.category == 'branded_food')
# (Add more filters as needed)

In [ ]:
# Prepare for correlation: select only nutrient columns, drop rows with too many missing values
from pyspark.sql.functions import col
min_valid = 3  # require at least 3 non-null nutrients
valid_row = sum([col(c).isNotNull().cast('int') for c in nutrient_cols]) >= min_valid
df_valid = df.where(valid_row)

# Collect to pandas for correlation matrix (PySpark does not have built-in corr matrix)
pdf = df_valid.select(nutrient_cols).toPandas()
corr_matrix = pdf.corr(method='pearson')
corr_matrix

In [ ]:
# Visualize correlation matrix as heatmap
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Nutrient Correlation Matrix")
plt.show()

In [ ]:
# Extract and display strong correlations (|r| > 0.5, off-diagonal)
strong_corrs = []
for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if i < j:
            val = corr_matrix.iloc[i, j]
            if abs(val) > 0.5:
                strong_corrs.append((col1, col2, val))

# Sort by absolute correlation
strong_corrs = sorted(strong_corrs, key=lambda x: -abs(x[2]))

print("Nutrient pairs with strong correlations (|r| > 0.5):")
for c1, c2, val in strong_corrs:
    print(f"{c1} - {c2}: r = {val:.2f}")

if not strong_corrs:
    print("No strong correlations found.")

### Interpretation
- Strong positive correlations may indicate nutrients that often appear together (e.g., sodium and protein in processed meats).
- Negative correlations may suggest substitutions or inverse effects (e.g., fiber vs. sugars in some products).
- These associations can be useful for product development, labeling, or nutritional research.